# Notebook 01 (version 3): Product data and choice sets

*Version 3: stricter filters (only the main headphone and drip-coffee-machine sub-categories; more accessory and manual-brewer words), the products removed by hand after run 2, and automatic balancing if one category cannot reach 25 sets.*

**Thesis:** *Evaluating Product Attribute Sensitivity in LLM-Based Shopping Agents*

This notebook prepares the product data for the experiment (Section 3.3 of the thesis). It:

1. downloads the product metadata for **headphones** (Electronics) and **coffee makers** (Home & Kitchen) from the Amazon Reviews 2023 dataset;
2. keeps only complete, realistic listings and removes duplicates, accessories and extreme prices;
3. marks which products have each **required feature** (used later for the constrained requests);
4. builds about **25 choice sets of 4 similar products per category** (about 50 sets, about 200 products);
5. saves everything to your Google Drive, together with a log of how many products survived each step (you need these numbers for Chapter 3/4).

**Runtime:** a normal CPU runtime is enough (no GPU needed). Runtime → Change runtime type → CPU.
The downloads are large (several GB), so the first run can take 20–40 minutes. Later runs reuse the downloaded files.

Run the cells from top to bottom.

## 1. Connect Google Drive and create the project folders
Everything is saved under `MyDrive/thesis-agents/`, so nothing is lost when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/thesis-agents'
RAW_DIR = '/content/raw'                    # big downloads stay on the Colab disk (temporary)
DATA_DIR = f'{PROJECT}/data'                # results are saved on Drive (permanent)
for d in [PROJECT, RAW_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)
print('Project folder:', PROJECT)

## 2. Settings
All choices that the thesis describes are collected here, so you can find and change them in one place.
If a category ends up with too few products, the cell 6 log will tell you which filter removed most of them; relax that setting here and run again.

In [ ]:
import re

SEED = 42                 # fixed random seed, so the same products and sets are chosen every time
SETS_PER_CATEGORY = 25    # target per category; 25 x 2 = 50 choice sets
TOTAL_SETS = 50           # if one category cannot reach 25 sets, the other one fills the gap
SET_SIZE = 4              # products per choice set
PRICE_BAND = 0.25         # every product within 25% of the set's median price (Section 3.3, step 4)
PRICE_TRIM = (0.01, 0.99) # remove prices outside the 1st to 99th percentile (Section 3.3, step 2)

# Words that are cut from titles before they are shown to the agents, because they act as extra cues
TITLE_NOISE = [r'\(discontinued by manufacturer\)', r'\(discontinued\)', r'\[retail packaging\]',
               r'-\s*amazon vine', r'\s-\s*new$']

# Products removed by hand after checking the sets (paste further parent_asin values here if you spot more)
EXCLUDE_ASINS = [
    # run 2, headphones: accessories and non-headphone items
    'B00G97RWPY', 'B0172U70BW', 'B0BHQK56JD', 'B01IRKSX0S', 'B07P54TLX8', 'B07BSWN112', 'B07BVFKGB1',
    'B083SGS81W', 'B005H3AVZE', 'B0BRNSP3Q2', 'B0BN9WM1FC',
    # run 2, coffee makers: covers, parts, urns, manual brewers and French presses
    'B003CGX9CK', 'B0C865P869', 'B08YS7CCBW', 'B08Z2ZXWS3', 'B077JKZ7T5', 'B081PLXZZR', 'B079GLQMRQ',
    'B0B8FKBMTP', 'B0149TSTDW', 'B09QBZN6PZ', 'B07J5F2L7H', 'B086H81KKS', 'B01FVMF7W2', 'B06XKNR7CL',
]

CATEGORIES = {
    'headphones': {
        'source': 'Electronics',
        'min_ratings': 20,
        'min_desc_words': 25,
        # the product's category path must contain one of these
        'category_match': ['Headphones'],
        # the title must contain one of these (keeps real headphones)
        'title_must': ['headphone', 'earbud', 'earphone'],
        # the title must match none of these patterns (accessories, multipacks, kids, business headsets, odd items)
        'title_exclude': [
            r'\bcase\b', r'\bcover\b', r'ear ?pads?', r'\bcushions?\b', r'\breplacement\b', r'\bcables?\b',
            r'\badapters?\b', r'\bstand\b', r'\bholder\b', r'ear ?tips', r'\bsticker', r'\bskin\b', r'\bsplitter\b',
            r'\bextension\b', r'\bcharger\b', r'\bdongle\b', r'\btransmitter\b',
            r'\bkids?\b', r'\bchildren\b', r'\btoddlers?\b', r'\bboys?\b', r'\bgirls?\b',
            r'\d+\s*-?\s*(pack|pcs|pieces)\b', r'\bpack of\b', r'\bbulk\b', r'\blot of\b', r'\bwholesale\b',
            r'\bclassroom\b', r'\bschool\b', r'\blibrary\b', r'\bmono\b', r'\bearset\b', r'\bsingle ear\b',
            r'\bsingle channel\b', r'\bsleep', r'\bheadband\b', r'\bwristband\b', r'\btv\b', r'\bassistive\b',
            r'\bir\b', r'\bparts?\b', r'\badditional\b', r'\brenewed\b', r'\brefurbished\b', r'cat.?s? ears?',
            r'\bunicorn\b', r'\belf\b', r'\bnovelty\b', r'\bmotorcycle\b', r'\bhelmet\b', r'\bswim',
            r'\bhooks?\b', r'\bloops?\b', r'\bgrips?\b', r'\bhat\b', r'\bbeanie\b', r'\battachment\b',
            r'\bplush\b', r'\bcartoon\b', r'\banimal\b', r'\bscanner\b', r'\bpocketalker\b',
            r'\badaptor\b', r'\bcleaner\b', r'\bcleaning\b', r'\bkit\b', r'single earbud'],
        # only these sub-categories are used
        'subcategory_keep': ['Over-Ear Headphones', 'On-Ear Headphones', 'Earbud Headphones'],
        # a 'headset' without 'headphone/earbud/earphone' in the title is usually an office or phone headset
        'drop_plain_headsets': True,
        # required features for the constrained requests (Section 3.5): name -> regex patterns
        'features': {
            'noise_cancelling': [r'noise[\s-]*cancel+ing', r'\banc\b'],
            'wireless':         [r'\bwireless\b', r'\bbluetooth\b'],
            'microphone':       [r'\bmicrophone\b', r'\bmics?\b', r'built-?in mic'],
        },
    },
    'coffee_makers': {
        'source': 'Home_and_Kitchen',
        'min_ratings': 10,       # fewer coffee makers have long descriptions, so these two are lower
        'min_desc_words': 10,
        'category_match': ['Coffee Makers', 'Coffee Machines'],
        'title_must': ['coffee maker', 'coffeemaker', 'coffee machine', 'coffee brewer', 'drip coffee'],
        # keeps electric drip coffee makers only
        'title_exclude': [
            r'\bfilters?\b', r'\bpods?\b', r'\bk-?cups?\b', r'\breplacement\b', r'\bdescal', r'\bcleaning\b',
            r'\bmugs?\b', r'\bfrother\b', r'\bscoop\b', r'\blid\b', r'\bparts?\b', r'\bdecanter\b', r'\bcapsules?\b',
            r'\bturkish\b', r'\bgreek\b', r'\bsiphon\b', r'\bsyphon\b', r'\bvacuum\b', r'\bcona\b', r'all-glass',
            r'pour[\s-]?over', r'\bdripper\b', r'\bespresso\b', r'\bcappuccino\b', r'cold\s?br(ew|uer)', r'\biced?\b',
            r'\bpercolator\b', r'\bperk\b', r'single[\s-]?(serve|cup)', r'\bkeurig\b', r'\btassimo\b', r'\bnespresso\b',
            r'\bmat\b', r'\brenewed\b', r'\brefurbished\b', r'\b220\b', r'\bjapan\b', r'\brechargeable\b',
            r'\bcordless\b', r'\bportable\b', r'on-the-go', r'\btravel\b', r'exclusively compatible',
            r'\b1[\s-]cup\b', r'\bone[\s-]cup\b', r'\bcamping\b', r'hand drip', r'\bbundle\b',
            r'\bcovers?\b', r'\btube\b', r'\bpump\b', r'french press', r'\bcafeti', r'\bpress\b', r'slow (drip|brew)',
            r'\bhario\b', r'case of', r'kompresso', r'hand carry', r'\burn\b', r'\b(2\d|[3-9]\d|1\d\d)[\s-]*cups?\b'],
        'subcategory_keep': ['Coffee Machines'],
        'drop_plain_headsets': False,
        'features': {
            'programmable':   [r'\bprogrammable\b', r'delay(ed)? brew', r'\btimer\b'],
            'thermal_carafe': [r'thermal carafe', r'\bthermal\b', r'stainless steel carafe'],
            'auto_shutoff':   [r'auto(matic)?[\s-]*(shut[\s-]*off|off)', r'auto shut'],
        },
    },
}
print('Settings loaded for:', list(CATEGORIES))

## 3. Download the raw metadata
The files come from the McAuley Lab (Hou et al., 2024a). The notebook first tries the Hugging Face copy and falls back to the university server if that fails.
Only the **metadata** files are downloaded (titles, prices, ratings, features, descriptions), not the reviews.

In [ ]:
import subprocess, gzip, shutil

HF_URL   = 'https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_{src}.jsonl'
UCSD_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_{src}.jsonl.gz'

def download(src):
    out = f'{RAW_DIR}/meta_{src}.jsonl'
    if os.path.exists(out) and os.path.getsize(out) > 1_000_000:
        print(f'{src}: already downloaded ({os.path.getsize(out)/1e9:.1f} GB)')
        return out
    print(f'{src}: downloading from Hugging Face ...')
    ok = subprocess.run(['wget', '-q', '-O', out, HF_URL.format(src=src)]).returncode == 0
    if not ok or os.path.getsize(out) < 1_000_000:
        print(f'{src}: Hugging Face failed, trying the UCSD server ...')
        gz = out + '.gz'
        subprocess.run(['wget', '-q', '-O', gz, UCSD_URL.format(src=src)], check=True)
        with gzip.open(gz, 'rb') as f_in, open(out, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(gz)
    print(f'{src}: done ({os.path.getsize(out)/1e9:.1f} GB)')
    return out

RAW_FILES = {cat: download(cfg['source']) for cat, cfg in CATEGORIES.items()}

## 4. Read the files and keep only candidate products
The raw files contain millions of products, too many to load at once. This cell reads them **line by line** and keeps only products in the right category, so it fits in Colab's memory. It also counts how many products are dropped at each step; the counts are saved at the end.

In [ ]:
import json
import pandas as pd

def as_text(x):
    # Amazon fields are sometimes a list of strings, sometimes a string, sometimes empty
    if x is None:
        return ''
    if isinstance(x, list):
        return ' '.join(str(s) for s in x if s)
    return str(x)

def parse_price(p):
    # prices appear as numbers, as strings like '29.99' or '$29.99', or as None / 'None'
    if p is None:
        return None
    if isinstance(p, (int, float)):
        return float(p) if p > 0 else None
    m = re.search(r'\d+(?:[.,]\d+)?', str(p).replace(',', ''))
    return float(m.group()) if m else None

def contains_any(text, words):
    t = text.lower()
    return any(w.lower() in t for w in words)

def matches_any(text, patterns):
    t = text.lower()
    return any(re.search(p, t) for p in patterns)

def clean_title(t):
    for p in TITLE_NOISE:
        t = re.sub(p, '', t, flags=re.IGNORECASE)
    return re.sub(r'\s{2,}', ' ', t).strip(' ,-')

def read_candidates(path, cfg, log):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            log['1_raw_products'] += 1
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            cats = r.get('categories') or []
            if not contains_any(' | '.join(cats), cfg['category_match']):
                continue
            log['2_in_category'] += 1
            rows.append({
                'parent_asin': r.get('parent_asin'),
                'title': as_text(r.get('title')).strip(),
                'store': as_text(r.get('store')).strip(),
                'price': parse_price(r.get('price')),
                'average_rating': r.get('average_rating'),
                'rating_number': r.get('rating_number'),
                'features': as_text(r.get('features')).strip(),
                'description': as_text(r.get('description')).strip(),
                'categories': ' > '.join(cats),
                'subcategory': cats[-1] if cats else '',
            })
    return pd.DataFrame(rows)

LOGS, POOLS = {}, {}
for cat, cfg in CATEGORIES.items():
    log = {'1_raw_products': 0, '2_in_category': 0}
    POOLS[cat] = read_candidates(RAW_FILES[cat], cfg, log)
    LOGS[cat] = log
    print(cat, log)

## 5. Clean the products
Steps, in this order (each step's count is logged):
1. title must name the product type and must not match the exclusion list (accessories, multipacks, kids' products, office headsets, refurbished items, non-drip coffee makers such as espresso, pour-over or Turkish); noise such as '(Discontinued by Manufacturer)' is cut from titles first;
2. complete data: a price, an average rating, enough ratings and a description of enough words (`min_ratings`, `min_desc_words` per category in cell 2);
3. remove duplicates (same product id, or the same title from the same store);
4. remove extreme prices outside the 1st to 99th percentile of the category.

In [ ]:
def clean(df, cfg, log):
    d = df.copy()
    d = d[~d['parent_asin'].isin(EXCLUDE_ASINS)]
    d['title'] = d['title'].apply(clean_title)
    def title_ok(t):
        if not contains_any(t, cfg['title_must']) or matches_any(t, cfg['title_exclude']):
            return False
        if cfg['drop_plain_headsets'] and re.search(r'\bheadset\b', t.lower()) \
                and not re.search(r'headphone|earbud|earphone', t.lower()):
            return False
        return True
    d = d[d['subcategory'].isin(cfg['subcategory_keep'])]
    d = d[d['title'].apply(title_ok)];                                     log['3_right_product_type'] = len(d)
    d['desc_words'] = d['description'].str.split().str.len().fillna(0)
    d = d[d['price'].notna() & d['average_rating'].notna()
          & (d['rating_number'].fillna(0) >= cfg['min_ratings'])
          & (d['desc_words'] >= cfg['min_desc_words'])];                           log['4_complete_data'] = len(d)
    d['title_key'] = d['title'].str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True).str.slice(0, 60)
    d = d.drop_duplicates('parent_asin').drop_duplicates(['store', 'title_key']); log['5_no_duplicates'] = len(d)
    lo, hi = d['price'].quantile(PRICE_TRIM[0]), d['price'].quantile(PRICE_TRIM[1])
    d = d[(d['price'] >= lo) & (d['price'] <= hi)];                         log['6_price_trimmed'] = len(d)
    log['price_range_kept'] = f'{lo:.2f} to {hi:.2f}'
    return d.drop(columns=['title_key']).reset_index(drop=True)

def add_feature_flags(d, cfg):
    text = (d['title'] + ' ' + d['features'] + ' ' + d['description']).str.lower()
    for name, patterns in cfg['features'].items():
        d['has_' + name] = text.apply(lambda t: any(re.search(p, t) for p in patterns))
    return d

CLEAN = {}
for cat, cfg in CATEGORIES.items():
    CLEAN[cat] = add_feature_flags(clean(POOLS[cat], cfg, LOGS[cat]), cfg)
    flags = [c for c in CLEAN[cat].columns if c.startswith('has_')]
    print(f'\n{cat}: {len(CLEAN[cat])} clean products')
    print('  share with each feature:', CLEAN[cat][flags].mean().round(2).to_dict())
    print('  log:', LOGS[cat])

## 6. Build the choice sets
A choice set is 4 products that are plausible answers to the same request (Section 3.3, step 4). For each category the code:
1. groups the clean products by sub-category (for example *Over-Ear Headphones*);
2. sorts them by price and looks at every run of 4 neighbouring products;
3. accepts a run only if all 4 prices are within 25% of their median, the 4 brands are different, and for at least one required feature the set contains a product **with** it and one **without** it (otherwise no constrained request could separate them later);
4. picks 25 of these runs at random (fixed seed) so that no product is used twice.

If one category cannot produce 25 sets without reusing products, the other category supplies the missing sets, so the total stays at 50. The printed line `possible sets ... -> sets used ...` shows what happened.

In [ ]:
import numpy as np

def candidate_sets(d, cfg):
    flags = ['has_' + f for f in cfg['features']]
    cands = []
    for sub, g in d.groupby('subcategory'):
        g = g.sort_values('price').reset_index(drop=True)
        for i in range(len(g) - SET_SIZE + 1):
            w = g.iloc[i:i + SET_SIZE]
            med = w['price'].median()
            if not ((w['price'] >= (1 - PRICE_BAND) * med) & (w['price'] <= (1 + PRICE_BAND) * med)).all():
                continue
            if w['store'].str.lower().nunique() < SET_SIZE:
                continue
            if not any(w[f].any() and (~w[f]).any() for f in flags):
                continue
            cands.append((sub, tuple(w['parent_asin'])))
    return cands

def pick_sets(cands, n, rng):
    order = rng.permutation(len(cands))
    used, chosen = set(), []
    for k in order:
        sub, asins = cands[k]
        if used.isdisjoint(asins):
            chosen.append((sub, asins)); used.update(asins)
        if len(chosen) == n:
            break
    return chosen

CANDS = {cat: candidate_sets(CLEAN[cat], cfg) for cat, cfg in CATEGORIES.items()}
# how many sets each category can give without reusing a product
possible = {cat: len(pick_sets(c, 10**6, np.random.default_rng(SEED))) for cat, c in CANDS.items()}
target = {cat: min(SETS_PER_CATEGORY, possible[cat]) for cat in CATEGORIES}
short = TOTAL_SETS - sum(target.values())
for cat in sorted(CATEGORIES, key=lambda c: -possible[c]):      # the larger category fills any gap
    extra = min(short, possible[cat] - target[cat]); target[cat] += extra; short -= extra
print('possible sets:', possible, '-> sets used:', target)
if sum(target.values()) < TOTAL_SETS:
    print(f'WARNING: only {sum(target.values())} sets in total. Lower min_ratings or min_desc_words, or raise PRICE_BAND, in cell 2.')

set_rows, product_rows = [], []
for cat, cfg in CATEGORIES.items():
    d = CLEAN[cat].set_index('parent_asin')
    chosen = pick_sets(CANDS[cat], target[cat], np.random.default_rng(SEED))
    LOGS[cat]['7_candidate_sets'] = len(CANDS[cat])
    LOGS[cat]['8_sets_chosen'] = len(chosen)
    for j, (sub, asins) in enumerate(chosen, start=1):
        set_id = f'{cat[:2].upper()}{j:02d}'
        set_rows.append({'set_id': set_id, 'category': cat, 'subcategory': sub,
                         'median_price': float(d.loc[list(asins), 'price'].median()),
                         **{f'product_{k+1}': a for k, a in enumerate(asins)}})
        for a in asins:
            product_rows.append({'set_id': set_id, 'category': cat, 'parent_asin': a, **d.loc[a].to_dict()})

choice_sets = pd.DataFrame(set_rows)
products = pd.DataFrame(product_rows)
print(f'{len(choice_sets)} choice sets, {len(products)} products')
choice_sets.groupby('category').size()

## 7. Save the results to Google Drive
Files written to `MyDrive/thesis-agents/data/`:
- `products_selected.csv`: the ~200 products used in the experiment (one row each, with its set id and feature flags);
- `choice_sets.csv`: the ~50 sets (which 4 products belong together);
- `clean_pool_<category>.parquet`: all clean products, in case a product has to be replaced later (Section 3.4);
- `data_preparation_log.json`: the counts after each step, for the thesis.

In [ ]:
products.to_csv(f'{DATA_DIR}/products_selected.csv', index=False)
choice_sets.to_csv(f'{DATA_DIR}/choice_sets.csv', index=False)
for cat in CATEGORIES:
    CLEAN[cat].to_parquet(f'{DATA_DIR}/clean_pool_{cat}.parquet', index=False)
settings = dict(SEED=SEED, SETS_PER_CATEGORY=SETS_PER_CATEGORY, TOTAL_SETS=TOTAL_SETS, PRICE_BAND=PRICE_BAND, PRICE_TRIM=PRICE_TRIM,
                EXCLUDE_ASINS=EXCLUDE_ASINS,
                per_category={c: {k: cfg[k] for k in ['min_ratings', 'min_desc_words', 'subcategory_keep', 'title_exclude']}
                              for c, cfg in CATEGORIES.items()})
with open(f'{DATA_DIR}/data_preparation_log.json', 'w') as f:
    json.dump({'settings': settings, 'counts': LOGS}, f, indent=2, default=str)
print('Saved to', DATA_DIR)
print(json.dumps(LOGS, indent=2, default=str))

## 8. Look at a few sets yourself
Read two or three sets and ask: *would a real shopper see these four as alternatives?* If a set looks odd (for example a kids' headset next to studio headphones), note its `set_id`; we can exclude it before building the trials.

In [ ]:
base = ['set_id', 'subcategory', 'store', 'title', 'price', 'average_rating', 'rating_number']
for cat, sid in choice_sets.groupby('category')['set_id'].first().items():
    flags = ['has_' + f for f in CATEGORIES[cat]['features']]
    print('\n' + '=' * 100)
    display(products.loc[products['set_id'] == sid, base + flags])